# 00. Official Pipeline Contract

이 노트북의 목적은 **학습이 아니다.**

지금까지 꼬였던 문제를 다시 만들지 않기 위해, 공식 CLRKDNet repo가 전제로 하는 입력/출력 계약을 먼저 확인한다.

확인할 것:

- 공식 config의 image size, `cut_height`, `sample_y`
- 학습 dataloader가 모델에 넣는 이미지 텐서의 색상 순서와 값 범위
- `.lines.txt` 라벨이 어떤 좌표계로 해석되는지
- 모델 출력이 어떤 tensor layout인지
- 공식 `get_lanes()`가 어떤 후처리 순서를 쓰는지
- 우리가 ONNX/Pi runtime에서 반드시 맞춰야 할 계약

이 노트북에서 확정한 내용은 이후 12 폴더의 모든 노트북이 따른다.

In [1]:
from pathlib import Path
import json
import re
import sys
from pprint import pprint

PROJECT_ROOT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization")
EXP_ROOT = PROJECT_ROOT / "10_experiments" / "12_clrkdnet_supervised_rebuild"
REPO_DIR = PROJECT_ROOT / "00_reference" / "repos" / "CLRKDNet"
RAW_LANE_DIR = PROJECT_ROOT / "20_shared_assets" / "dataset" / "lane"
OUT_DIR = EXP_ROOT / "review_outputs" / "00_official_pipeline_contract"
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "EXP_ROOT": EXP_ROOT,
    "REPO_DIR": REPO_DIR,
    "RAW_LANE_DIR": RAW_LANE_DIR,
    "OUT_DIR": OUT_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())

assert REPO_DIR.exists(), "공식 CLRKDNet repo 경로가 없습니다."
assert RAW_LANE_DIR.exists(), "20_shared_assets/dataset/lane 경로가 없습니다."

PROJECT_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization
  exists: True
EXP_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild
  exists: True
REPO_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\00_reference\repos\CLRKDNet
  exists: True
RAW_LANE_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\dataset\lane
  exists: True
OUT_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\00_official_pipeline_contract
  exists: True


## 1. 공식 config에서 기본 좌표계를 확인한다

CLRKDNet의 CULane config는 모델이 어떤 크기의 이미지를 전제로 하는지, 그리고 원본 이미지에서 위쪽을 얼마나 자르는지 정의한다.

우리 프로젝트에서는 카메라 원본이 `1296x972`였고, 기존 실험에서 `cut_height=445`, 모델 입력 `800x320`을 사용했다. 하지만 먼저 공식 config가 어떤 값을 쓰는지 확인한다.

In [2]:
CONFIG_PATH = REPO_DIR / "configs" / "ResNet18_CULane.py"
assert CONFIG_PATH.exists(), CONFIG_PATH
config_text = CONFIG_PATH.read_text(encoding="utf-8")

keys = ["num_points", "max_lanes", "sample_y", "ori_img_w", "ori_img_h", "img_w", "img_h", "cut_height"]
for key in keys:
    m = re.search(rf"^\s*{key}\s*=\s*(.+)$", config_text, flags=re.MULTILINE)
    print(f"{key:>12}:", m.group(1).strip() if m else "<not found>")

print("\nConfig path:", CONFIG_PATH)

  num_points: 72
   max_lanes: 4
    sample_y: range(589, 230, -20)
   ori_img_w: 1640
   ori_img_h: 590
       img_w: 800
       img_h: 320
  cut_height: 270

Config path: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\00_reference\repos\CLRKDNet\configs\ResNet18_CULane.py


### 결과 해석

공식 `ResNet18_CULane.py`는 CULane 원본 기준 설정이다.

- 공식 CULane 원본 크기: `1640x590`
- 공식 crop: `cut_height=270`
- 모델 입력 크기: `800x320`
- lane 좌표 샘플 수: `num_points=72`
- 최대 lane 수: `max_lanes=4`

중요한 점은, 이 값이 우리 Pi 카메라 설정과 다르다는 것이다. 우리 프로젝트는 기존 실험에서 원본 `1296x972`, `cut_height=445`, 모델 입력 `800x320`을 썼다. 따라서 12 실험에서도 공식 config를 그대로 쓰는 게 아니라, **모델 구조와 학습 방식은 공식대로 두고 이미지 geometry만 우리 카메라에 맞게 patch**해야 한다.

즉 바꿔야 하는 것은 모델 철학이 아니라 좌표계다.

## 2. 공식 학습 transform의 입력 range를 확인한다

중요한 지점이다. 공식 repo의 `GenerateLaneLine` transform은 이미지를 모델 텐서로 바꾸기 직전에 `/255`를 수행한다.

즉 config에 `Normalize`가 따로 없어도, 학습 시 모델이 보는 이미지는 일반적으로 **BGR float32, 범위 [0, 1]** 이어야 한다.

이 계약이 ONNX export와 Pi runtime에서 어긋나면, 학습이 잘 되어도 실제 주행에서 모델이 엉망으로 보일 수 있다.

In [3]:
GEN_PATH = REPO_DIR / "clrkd" / "datasets" / "process" / "generate_lane_line.py"
assert GEN_PATH.exists(), GEN_PATH
text = GEN_PATH.read_text(encoding="utf-8")

for i, line in enumerate(text.splitlines(), start=1):
    if "/ 255" in line or "astype(np.float32)" in line or "ToTensor" in line:
        start = max(1, i - 4)
        end = min(len(text.splitlines()), i + 4)
        print(f"\n--- generate_lane_line.py around line {i} ---")
        for j, ctx in enumerate(text.splitlines()[start-1:end], start=start):
            marker = ">>" if j == i else "  "
            print(f"{marker} {j:04d}: {ctx}")


--- generate_lane_line.py around line 211 ---
   0207:                     self.logger.critical(
   0208:                         'Transform annotation failed 30 times :(')
   0209:                     exit()
   0210: 
>> 0211:         sample['img'] = img.astype(np.float32) / 255.
   0212:         sample['lane_line'] = label
   0213:         sample['lanes_endpoints'] = lane_endpoints
   0214:         sample['gt_points'] = new_anno['lanes']
   0215:         sample['seg'] = seg.get_arr() if self.training else np.zeros(


### 결과 해석

가장 중요한 결과다. 공식 학습 transform 안에서 이미지가 다음처럼 바뀐다.

```python
sample['img'] = img.astype(np.float32) / 255.
```

즉 CLRKDNet은 학습 때 `[0, 255]` 이미지가 아니라 **`[0, 1]` float32 이미지**를 본다. 이 정보는 config 파일 바깥의 transform 코드 안에 숨어 있어서 놓치기 쉽다.

따라서 앞으로 모든 export/runtime/ONNX/Pi 코드에서 반드시 확인해야 할 계약은 다음이다.

```text
입력 이미지 → crop/resize → float32 → /255 → NCHW
```

예전 Pi runtime에서 `/255`가 빠졌던 것이 의심되는 이유도 바로 이 지점이다.

## 3. 공식 prior는 어떤 선을 상정하는가

CLRKDNet은 임의의 선분 detector가 아니라, lane을 `x = f(y)` 형태로 표현한다.

공식 head는 prior를 `[start_y, start_x, theta]`로 초기화하고, 이후 각 y 샘플 위치에서 x offset을 예측한다. 따라서 가로선이나 종료선처럼 한 y에서 x가 길게 퍼지는 구조는 CLRKDNet의 주 대상이 아니다.

이 셀에서는 공식 prior 초기화 코드를 읽어, left/bottom/right prior가 어떻게 만들어지는지 확인한다.

In [4]:
HEAD_PATH = REPO_DIR / "clrkd" / "models" / "heads" / "clr_head.py"
assert HEAD_PATH.exists(), HEAD_PATH
head_lines = HEAD_PATH.read_text(encoding="utf-8").splitlines()

# print _init_prior_embeddings and generate_priors snippets
for keyword in ["def generate_priors_from_embeddings", "def _init_prior_embeddings"]:
    idx = next(i for i, line in enumerate(head_lines) if keyword in line)
    print(f"\n--- clr_head.py: {keyword} ---")
    for j in range(idx, min(idx + 45, len(head_lines))):
        print(f"{j+1:04d}: {head_lines[j]}")


--- clr_head.py: def generate_priors_from_embeddings ---
0124:     def generate_priors_from_embeddings(self):
0125:         predictions = self.prior_embeddings.weight  # (num_prop, 3)
0126: 
0127:         # 2 scores, 1 start_y, 1 start_x, 1 theta, 1 length, 72 coordinates, score[0] = negative prob, score[1] = positive prob
0128:         priors = predictions.new_zeros(
0129:             (self.num_priors, 2 + 2 + 2 + self.n_offsets), device=predictions.device)
0130: 
0131:         priors[:, 2:5] = predictions.clone()
0132:         priors[:, 6:] = (
0133:             priors[:, 3].unsqueeze(1).clone().repeat(1, self.n_offsets) *
0134:             (self.img_w - 1) +
0135:             ((1 - self.prior_ys.repeat(self.num_priors, 1) -
0136:               priors[:, 2].unsqueeze(1).clone().repeat(1, self.n_offsets)) *
0137:              self.img_h / torch.tan(priors[:, 4].unsqueeze(1).clone().repeat(
0138:                  1, self.n_offsets) * math.pi + 1e-5))) / (self.img_w - 1)
0139: 
0140:  

### 결과 해석

공식 head는 lane 후보를 `start_y`, `start_x`, `theta`를 가진 prior로 초기화한다. prior는 크게 세 위치에서 시작한다.

- 왼쪽 edge에서 시작하는 후보
- 아래쪽 bottom에서 시작하는 후보
- 오른쪽 edge에서 시작하는 후보

그리고 각 후보는 여러 y 위치에서의 x 좌표들을 예측하는 방식으로 lane이 된다. 즉 CLRKDNet은 일반적인 “임의의 선분 탐지기”가 아니라, **진행 방향 lane을 `x = f(y)` 형태로 보는 모델**이다.

그래서 종료선처럼 거의 가로인 선, 짧은 노란 조각, y 방향 길이가 거의 없는 선은 학습 라벨로 넣으면 모델 표현과 잘 맞지 않는다. 12의 라벨 정책에서 이런 가로선/짧은 조각을 제외해야 하는 이유가 여기에 있다.

## 4. 공식 `get_lanes()` 후처리 순서를 확인한다

공식 evaluation/inference는 단순히 score top-k를 고르는 것이 아니다.

핵심 순서:

1. class logits에 softmax 적용
2. lane confidence threshold 적용
3. LineIoU NMS 적용
4. top-k 적용
5. normalized output을 원본 이미지 좌표의 lane point로 변환

이 순서가 ONNX runtime에서 빠지면 같은 lane의 중복 anchor가 여러 개 살아남을 수 있다.

In [5]:
for keyword in ["def get_lanes", "def predictions_to_pred"]:
    matches = [i for i, line in enumerate(head_lines) if keyword in line]
    if not matches:
        print(keyword, "not found")
        continue
    idx = matches[0]
    print(f"\n--- clr_head.py: {keyword} ---")
    for j in range(idx, min(idx + 75, len(head_lines))):
        print(f"{j+1:04d}: {head_lines[j]}")


--- clr_head.py: def get_lanes ---
0459:     def get_lanes(self, output, as_lanes=True):
0460:         '''
0461:         Convert model output to lanes.
0462:         '''
0463:         softmax = nn.Softmax(dim=1)
0464: 
0465:         decoded = []
0466:         for predictions in output:
0467:             # filter out the conf lower than conf threshold
0468:             threshold = self.cfg.test_parameters.conf_threshold
0469:             scores = softmax(predictions[:, :2])[:, 1]
0470:             keep_inds = scores >= threshold
0471:             predictions = predictions[keep_inds]
0472:             scores = scores[keep_inds]
0473: 
0474:             if predictions.shape[0] == 0:
0475:                 decoded.append([])
0476:                 continue
0477:             nms_predictions = predictions.detach().clone()
0478:             nms_predictions = torch.cat(
0479:                 [nms_predictions[..., :4], nms_predictions[..., 5:]], dim=-1)
0480:             nms_predictions[..., 4] 

### 결과 해석

공식 `get_lanes()`는 단순히 confidence 높은 후보 4개를 고르지 않는다. 순서는 다음이다.

```text
softmax
→ confidence threshold
→ LineIoU NMS
→ max_lanes top-k
→ predictions_to_pred
```

여기서 `LineIoU NMS`가 중요하다. 같은 lane을 여러 prior가 비슷하게 예측하면, NMS가 중복 후보를 줄여준다.

예전 ONNX/Pi runtime에서 score top-k만 사용했다면, 같은 차선 후보가 여러 개 살아남을 수 있다. 그러면 후처리에서는 “왼쪽/오른쪽 lane을 찾는 문제”가 아니라 “중복 anchor 더미에서 적당한 걸 고르는 문제”가 되어 버린다.

따라서 12에서는 먼저 PyTorch 공식 `get_lanes()`를 기준으로 삼고, ONNX decoder는 나중에 이 결과와 parity를 맞춰야 한다.

## 5. 공식 `.lines.txt` 라벨의 의미를 확인한다

CULane 라벨은 이미지 하나에 대응되는 `.lines.txt` 파일이다.

- 한 줄 = lane line 하나
- 한 줄 안의 값 = `x1 y1 x2 y2 ...`
- 좌표는 원본 이미지 좌표계 기준
- CLRKDNet transform은 이 좌표를 crop/resize/augmentation 후 내부 target으로 바꾼다

다음 셀에서는 공식 dataset reader가 `.lines.txt`를 어떻게 읽는지 확인한다.

In [6]:
CULANE_DATASET_PATH = REPO_DIR / "clrkd" / "datasets" / "culane.py"
assert CULANE_DATASET_PATH.exists(), CULANE_DATASET_PATH
culane_lines = CULANE_DATASET_PATH.read_text(encoding="utf-8").splitlines()

interesting = ["lines.txt", "load_annotation", "get_data_info", "train_gt.txt", "list"]
for i, line in enumerate(culane_lines, start=1):
    if any(k in line for k in interesting):
        start = max(1, i - 5)
        end = min(len(culane_lines), i + 10)
        print(f"\n--- culane.py around line {i} ---")
        for j in range(start, end + 1):
            print(f"{j:04d}: {culane_lines[j-1]}")


--- culane.py around line 13 ---
0008: from tqdm import tqdm
0009: import logging
0010: import pickle as pkl
0011: 
0012: LIST_FILE = {
0013:     'train': 'list/train_gt.txt',
0014:     'val': 'list/val.txt',
0015:     'test': 'list/test.txt',
0016: }
0017: 
0018: CATEGORYS = {
0019:     'normal': 'list/test_split/test0_normal.txt',
0020:     'crowd': 'list/test_split/test1_crowd.txt',
0021:     'hlight': 'list/test_split/test2_hlight.txt',
0022:     'shadow': 'list/test_split/test3_shadow.txt',
0023:     'noline': 'list/test_split/test4_noline.txt',

--- culane.py around line 14 ---
0009: import logging
0010: import pickle as pkl
0011: 
0012: LIST_FILE = {
0013:     'train': 'list/train_gt.txt',
0014:     'val': 'list/val.txt',
0015:     'test': 'list/test.txt',
0016: }
0017: 
0018: CATEGORYS = {
0019:     'normal': 'list/test_split/test0_normal.txt',
0020:     'crowd': 'list/test_split/test1_crowd.txt',
0021:     'hlight': 'list/test_split/test2_hlight.txt',
0022:     'shadow': 'lis

### 결과 해석

공식 CULane dataset reader는 split 파일을 다음처럼 기대한다.

```text
list/train_gt.txt
list/val.txt
list/test.txt
```

그리고 평가용 category로 `normal`, `curve`, `cross`, `night` 같은 파일을 따로 둔다. 이 category들은 모델이 학습하는 클래스가 아니라, **평가할 때 장면별 성능을 나눠 보기 위한 목록**이다.

우리 프로젝트에서 꼭 scene class를 만들 필요는 없다. 필요한 것은 이미지마다 `.lines.txt`가 있고, list 파일들이 그 이미지/라벨을 공식 규약대로 가리키는 것이다.

즉 우리의 학습 데이터도 본질은 다음 구조면 된다.

```text
이미지 1장 + 대응되는 .lines.txt 1개
```

## 6. 현재 공유 raw 데이터 위치만 확인한다

이 노트북에서는 아직 라벨을 만들지 않는다. 단지 12 실험이 읽을 raw 데이터가 어디에 있는지 확인한다.

In [7]:
for sub in ["raw", "holdout"]:
    p = RAW_LANE_DIR / sub
    print("\n", p)
    if not p.exists():
        print("  <missing>")
        continue
    for child in sorted(p.iterdir()):
        if child.is_dir():
            img_count = sum(1 for ext in ("*.jpg", "*.jpeg", "*.png") for _ in child.rglob(ext))
            print(f"  {child.name:20s} images={img_count}")


 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\dataset\lane\raw
  field1               images=4105
  field2               images=5360
  field3               images=1810

 ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\dataset\lane\holdout
  background_field1    images=500
  background_field2    images=319


### 결과 해석

현재 shared raw 데이터 집계는 다음과 같다.

```text
raw/field1: 4105장
raw/field2: 5360장
raw/field3: 1810장
holdout/background_field1: 500장
holdout/background_field2: 319장
```

학습 후보 raw는 충분하다. 이제 핵심은 개수가 아니라 **어떤 프레임에 신뢰 가능한 lane label을 만들 수 있느냐**다.

특히 background holdout은 학습에 섞지 않는 것이 좋다. 이 데이터는 실제 맵 순회 시퀀스에 가깝기 때문에, 나중에 fine-tuning 결과가 “주행처럼 들어오는 이미지 흐름”에서 안정적인지 확인하는 테스트셋으로 가치가 크다.

## 7. 계약 초안 저장

다음 셀은 지금까지 확인한 내용을 `official_pipeline_contract.json`으로 저장한다. 이후 노트북은 이 파일을 기준으로 계약을 어겼는지 확인한다.

In [8]:
contract = {
    "source": "official CLRKDNet repo inspection",
    "repo_dir": str(REPO_DIR),
    "config_path": str(CONFIG_PATH),
    "input_contract": {
        "color_order_before_tensor": "BGR, because cv2.imread is used in dataset pipeline unless explicitly converted",
        "dtype": "float32",
        "range": "[0, 1] after img.astype(np.float32) / 255.",
        "layout": "NCHW after ToTensor",
        "model_input_size": {"height": 320, "width": 800},
        "project_raw_size_expected": {"height": 972, "width": 1296},
        "project_cut_height_previous": 445,
    },
    "label_contract": {
        "format": "CULane .lines.txt",
        "one_line_means": "one lane polyline",
        "coordinate_basis": "original image coordinates before model crop/resize",
        "not_good_targets": ["horizontal finish lines", "nearly horizontal stop lines", "short yellow fragments with low y-span"],
    },
    "official_postprocess_contract": {
        "steps": ["softmax", "confidence threshold", "LineIoU NMS", "top-k", "predictions_to_pred"],
        "warning": "ONNX/Pi runtime must not use score-only top-k as final decoder unless parity is explicitly measured.",
    },
    "next_notebook": "01_raw_data_inventory.ipynb",
}

out_path = OUT_DIR / "official_pipeline_contract.json"
out_path.write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", out_path)
pprint(contract)

saved: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\00_official_pipeline_contract\official_pipeline_contract.json
{'config_path': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\00_reference\\repos\\CLRKDNet\\configs\\ResNet18_CULane.py',
 'input_contract': {'color_order_before_tensor': 'BGR, because cv2.imread is '
                                                 'used in dataset pipeline '
                                                 'unless explicitly converted',
                    'dtype': 'float32',
                    'layout': 'NCHW after ToTensor',
                    'model_input_size': {'height': 320, 'width': 800},
                    'project_cut_height_previous': 445,
                    'project_raw_size_expected': {'height': 972, 'width': 1296},
                    'range': '[0, 1] after img.astype(np.float32) / 255.'},
 'label_contract': {'coordinate_basis': '

### 결과 해석

`official_pipeline_contract.json`이 저장되었다. 이 파일은 이후 노트북들이 따라야 할 계약 메모다.

지금 확정한 핵심은 다음이다.

```text
학습 입력: BGR 기반 float32, /255 후 [0, 1]
모델 입력 크기: 800x320
라벨 형식: CULane .lines.txt
좋지 않은 라벨: 가로선, 종료선, y-span이 짧은 조각
공식 후처리: softmax + threshold + LineIoU NMS + top-k
```

다만 이 JSON은 “최종 진리”라기보다 12 실험의 시작 계약이다. 다음 노트북들에서 실제 dataloader 샘플, 라벨 overlay, ONNX parity를 확인하면서 필요하면 더 엄밀하게 업데이트한다.

## 이번 노트북에서 확정할 내용

실행 후 아래를 확인한다.

- 공식 학습 입력은 BGR float32 `[0, 1]`인지
- 공식 prior가 가로선이 아니라 진행 방향 lane을 전제로 하는지
- 공식 후처리가 LineIoU NMS를 포함하는지
- `.lines.txt`가 원본 이미지 좌표계의 lane polyline인지

## 아직 의심스러운 것

- 우리 field1/2/3 raw에서 어떤 노란선 조각을 lane label로 인정할지
- field1/2 HSV pseudo label과 field3 autolabel이 얼마나 다른지
- 기존 ONNX/Pi runtime이 공식 postprocess와 얼마나 달랐는지

## 다음 단계

`01_raw_data_inventory.ipynb`에서 `20_shared_assets/dataset/lane/raw`와 `holdout`에 모은 데이터를 실제로 집계한다.
라벨 생성은 아직 하지 않는다.